In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import ast
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [3]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [4]:
drive_root = "/content/drive/MyDrive"

matches = []

for root, dirs, files in os.walk(drive_root):
    for f in files:
        if f in [
            "scenario23_img_beam.csv",
            "scenario23_pos_beam.csv",
            "scenario23.csv"
        ]:
            matches.append(os.path.join(root, f))

print("Found full CSV files:")
for m in matches:
    print(m)

Found full CSV files:
/content/drive/MyDrive/Image beam/scenario23_img_beam.csv
/content/drive/MyDrive/Pos beam/scenario23_pos_beam.csv


In [5]:
FULL_CSV_PATH = "/content/drive/MyDrive/Pos beam/scenario23_pos_beam.csv"


full_df = pd.read_csv(FULL_CSV_PATH)

print("Shape:", full_df.shape)
display(full_df.head())
print("Columns:", full_df.columns.tolist())

Shape: (11387, 3)


,index,unit2_pos,unit1_beam
0,1,"[0.3195560203421424, 0.4874879026449761]",22
1,2,"[0.3176783761879752, 0.48679662657070005]",22
2,3,"[0.31603840851137044, 0.48610535045712955]",22
3,4,"[0.31458858201343814, 0.4854140743828535]",22
4,5,"[0.3123544231513711, 0.48444628784743143]",20


Columns: ['index', 'unit2_pos', 'unit1_beam']


In [6]:
required_cols = ["index", "unit2_pos", "unit1_beam"]

for col in required_cols:
    assert col in full_df.columns, f"Missing column: {col}"

full_df = full_df.sort_values("index").reset_index(drop=True)

print(full_df[required_cols].head())
print(full_df[required_cols].tail())

print("Total samples:", len(full_df))
print("Index min:", full_df["index"].min())
print("Index max:", full_df["index"].max())

   index                                   unit2_pos  unit1_beam
0      1    [0.3195560203421424, 0.4874879026449761]          22
1      2   [0.3176783761879752, 0.48679662657070005]          22
2      3  [0.31603840851137044, 0.48610535045712955]          22
3      4   [0.31458858201343814, 0.4854140743828535]          22
4      5   [0.3123544231513711, 0.48444628784743143]          20
       index                                  unit2_pos  unit1_beam
11382  11383  [0.33025146171424064, 0.4216784183738069]          20
11383  11384   [0.3300137852366781, 0.4216784183738069]          19
11384  11385   [0.3298236440580057, 0.4216784183738069]          19
11385  11386   [0.3293482911028806, 0.4218166735847326]          19
11386  11387  [0.32922945287254335, 0.4219549287956584]          19
Total samples: 11387
Index min: 1
Index max: 11387


In [7]:
idx = full_df["index"].astype(int).values
diffs = np.diff(idx)

unique_diffs, counts = np.unique(diffs, return_counts=True)

print("Unique index differences:", unique_diffs, counts)
print("Continuous +1 transitions:", np.sum(diffs == 1))
print("Broken transitions:", np.sum(diffs != 1))

Unique index differences: [1] [11386]
Continuous +1 transitions: 11386
Broken transitions: 0


position feature function

In [8]:
def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value


def add_position_features(df):
    df = df.copy()

    parsed = df["unit2_pos"].apply(parse_unit2_pos)

    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    eps = 1e-8

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["distance2"] = df["distance"] ** 2
    df["distance3"] = df["distance"] ** 3

    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_x3"] = df["pos_x"] ** 3
    df["pos_y3"] = df["pos_y"] ** 3

    df["pos_xy"] = df["pos_x"] * df["pos_y"]

    df["unit_x"] = df["pos_x"] / (df["distance"] + eps)
    df["unit_y"] = df["pos_y"] / (df["distance"] + eps)

    df["sin2_angle"] = np.sin(2 * df["angle"])
    df["cos2_angle"] = np.cos(2 * df["angle"])
    df["sin3_angle"] = np.sin(3 * df["angle"])
    df["cos3_angle"] = np.cos(3 * df["angle"])

    df["dist_sin"] = df["distance"] * df["sin_angle"]
    df["dist_cos"] = df["distance"] * df["cos_angle"]

    return df

In [9]:
full_df_fe = add_position_features(full_df)

feature_cols = [
    "pos_x", "pos_y",
    "distance", "distance2", "distance3",
    "angle",
    "sin_angle", "cos_angle",
    "pos_x2", "pos_y2",
    "pos_x3", "pos_y3",
    "pos_xy",
    "unit_x", "unit_y",
    "sin2_angle", "cos2_angle",
    "sin3_angle", "cos3_angle",
    "dist_sin", "dist_cos"
]

label_col = "unit1_beam"

print("Feature count:", len(feature_cols))
print("Label unique:", full_df_fe[label_col].nunique())
print("Label min/max:", full_df_fe[label_col].min(), full_df_fe[label_col].max())

display(full_df_fe[["index", "unit2_pos", *feature_cols, label_col]].head())

Feature count: 21
Label unique: 29
Label min/max: 2 30


,index,unit2_pos,pos_x,pos_y,distance,distance2,distance3,angle,sin_angle,cos_angle,...,pos_xy,unit_x,unit_y,sin2_angle,cos2_angle,sin3_angle,cos3_angle,dist_sin,dist_cos,unit1_beam
0,1,"[0.3195560203421424, 0.4874879026449761]",0.319556,0.487488,0.582890,0.339761,0.198043,0.990553,0.836329,0.548227,...,0.155780,0.548227,0.836329,0.916997,-0.398894,0.169116,-0.985596,0.487488,0.319556,22
1,2,"[0.3176783761879752, 0.48679662657070005]",0.317678,0.486797,0.581283,0.337891,0.196410,0.992603,0.837451,0.546512,...,0.154645,0.546512,0.837451,0.915354,-0.402649,0.163053,-0.986617,0.486797,0.317678,22
2,3,"[0.31603840851137044, 0.48610535045712955]",0.316038,0.486105,0.579809,0.336179,0.194919,0.994320,0.838389,0.545073,...,0.153628,0.545073,0.838388,0.913966,-0.405791,0.157968,-0.987444,0.486105,0.316038,22
3,4,"[0.31458858201343814, 0.4854140743828535]",0.314589,0.485414,0.578440,0.334593,0.193542,0.995770,0.839178,0.543857,...,0.152706,0.543857,0.839178,0.912785,-0.408439,0.153671,-0.988122,0.485414,0.314589,22
4,5,"[0.3123544231513711, 0.48444628784743143]",0.312354,0.484446,0.576414,0.332253,0.191516,0.998109,0.840448,0.541892,...,0.151319,0.541892,0.840448,0.910864,-0.412706,0.146733,-0.989176,0.484446,0.312354,20


In [10]:
all_labels = sorted(full_df_fe[label_col].astype(int).unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes = len(all_labels)

print("num_classes:", num_classes)
print("label_to_id:", label_to_id)

num_classes: 29
label_to_id: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3, np.int64(6): 4, np.int64(7): 5, np.int64(8): 6, np.int64(9): 7, np.int64(10): 8, np.int64(11): 9, np.int64(12): 10, np.int64(13): 11, np.int64(14): 12, np.int64(15): 13, np.int64(16): 14, np.int64(17): 15, np.int64(18): 16, np.int64(19): 17, np.int64(20): 18, np.int64(21): 19, np.int64(22): 20, np.int64(23): 21, np.int64(24): 22, np.int64(25): 23, np.int64(26): 24, np.int64(27): 25, np.int64(28): 26, np.int64(29): 27, np.int64(30): 28}


Paper style sequence build 

In [11]:
def build_strict_sequences_from_full_df(df, feature_cols, label_col, label_to_id, seq_len=4):
    df = df.sort_values("index").reset_index(drop=True).copy()

    idx_values = df["index"].astype(int).values
    X_raw = df[feature_cols].astype(float).values.astype(np.float32)
    y_raw = df[label_col].astype(int).values

    X_seq = []
    y_seq = []
    target_indices = []

    for i in range(seq_len - 1, len(df) - 1):
        past_idx = idx_values[i - seq_len + 1 : i + 1]
        target_idx = idx_values[i + 1]

        all_idx = np.concatenate([past_idx, [target_idx]])
        expected_idx = np.arange(all_idx[0], all_idx[0] + seq_len + 1)

        if np.array_equal(all_idx, expected_idx):
            X_seq.append(X_raw[i - seq_len + 1 : i + 1])
            y_seq.append(label_to_id[int(y_raw[i + 1])])
            target_indices.append(target_idx)

    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.int64)
    target_indices = np.array(target_indices, dtype=np.int64)

    return X_seq, y_seq, target_indices


SEQ_LEN = 4

X_seq_all, y_seq_all, target_indices_all = build_strict_sequences_from_full_df(
    full_df_fe,
    feature_cols,
    label_col,
    label_to_id,
    seq_len=SEQ_LEN
)

print("X_seq_all:", X_seq_all.shape)
print("y_seq_all:", y_seq_all.shape)
print("target_indices_all:", target_indices_all.shape)

print("First target indices:", target_indices_all[:10])
print("Last target indices:", target_indices_all[-10:])

X_seq_all: (11383, 4, 21)
y_seq_all: (11383,)
target_indices_all: (11383,)
First target indices: [ 5  6  7  8  9 10 11 12 13 14]
Last target indices: [11378 11379 11380 11381 11382 11383 11384 11385 11386 11387]


stratified split 

In [12]:
indices = np.arange(len(X_seq_all))

idx_train, idx_temp, y_train_temp, y_temp = train_test_split(
    indices,
    y_seq_all,
    test_size=0.30,
    random_state=SEED,
    stratify=y_seq_all
)

idx_val, idx_test, y_val_temp, y_test_temp = train_test_split(
    idx_temp,
    y_temp,
    test_size=1/3,
    random_state=SEED,
    stratify=y_temp
)

X_train_seq_raw = X_seq_all[idx_train]
y_train_seq_raw = y_seq_all[idx_train]

X_val_seq_raw = X_seq_all[idx_val]
y_val_seq_raw = y_seq_all[idx_val]

X_test_seq_raw = X_seq_all[idx_test]
y_test_seq_raw = y_seq_all[idx_test]

print("Train:", X_train_seq_raw.shape, y_train_seq_raw.shape)
print("Val  :", X_val_seq_raw.shape, y_val_seq_raw.shape)
print("Test :", X_test_seq_raw.shape, y_test_seq_raw.shape)

print("Train classes:", len(np.unique(y_train_seq_raw)))
print("Val classes  :", len(np.unique(y_val_seq_raw)))
print("Test classes :", len(np.unique(y_test_seq_raw)))

Train: (7968, 4, 21) (7968,)
Val  : (2276, 4, 21) (2276,)
Test : (1139, 4, 21) (1139,)
Train classes: 29
Val classes  : 29
Test classes : 29


In [13]:
def show_label_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)

    df = pd.DataFrame({
        "class_id": unique,
        "count": counts,
        "percent": 100 * counts / counts.sum()
    })

    print("\n" + "=" * 60)
    print(name)
    print("samples:", len(y))
    print("unique classes:", len(unique))
    display(df.sort_values("count", ascending=False).head(20))

    return set(unique)


train_classes = show_label_distribution("TRAIN", y_train_seq_raw)
val_classes   = show_label_distribution("VAL", y_val_seq_raw)
test_classes  = show_label_distribution("TEST", y_test_seq_raw)

print("Test classes missing from train:", sorted(test_classes - train_classes))
print("Val classes missing from train :", sorted(val_classes - train_classes))


TRAIN
samples: 7968
unique classes: 29


,class_id,count,percent
15,15,1982,24.874498
13,13,1110,13.930723
14,14,678,8.509036
12,12,580,7.279116
17,17,484,6.074297
18,18,462,5.798193
10,10,421,5.283635
9,9,309,3.878012
0,0,237,2.974398
2,2,232,2.911647



VAL
samples: 2276
unique classes: 29


,class_id,count,percent
15,15,566,24.868190
13,13,316,13.884007
14,14,194,8.523726
12,12,166,7.293497
17,17,138,6.063269
18,18,132,5.799649
10,10,120,5.272408
9,9,89,3.910369
2,2,67,2.943761
0,0,67,2.943761



TEST
samples: 1139
unique classes: 29


,class_id,count,percent
15,15,283,24.846356
13,13,159,13.959614
14,14,97,8.516242
12,12,83,7.287094
17,17,70,6.145742
18,18,66,5.794557
10,10,60,5.267779
9,9,44,3.863038
0,0,34,2.985075
2,2,33,2.897278


Test classes missing from train: []
Val classes missing from train : []


train only normalization

In [14]:
train_flat = X_train_seq_raw.reshape(-1, X_train_seq_raw.shape[-1])

seq_mean = train_flat.mean(axis=0)
seq_std = train_flat.std(axis=0)
seq_std[seq_std == 0] = 1.0

X_train_seq = (X_train_seq_raw - seq_mean) / seq_std
X_val_seq   = (X_val_seq_raw - seq_mean) / seq_std
X_test_seq  = (X_test_seq_raw - seq_mean) / seq_std

X_train_seq = torch.tensor(X_train_seq, dtype=torch.float32)
X_val_seq   = torch.tensor(X_val_seq, dtype=torch.float32)
X_test_seq  = torch.tensor(X_test_seq, dtype=torch.float32)

y_train_seq = torch.tensor(y_train_seq_raw, dtype=torch.long)
y_val_seq   = torch.tensor(y_val_seq_raw, dtype=torch.long)
y_test_seq  = torch.tensor(y_test_seq_raw, dtype=torch.long)

print("X_train_seq:", X_train_seq.shape)
print("X_val_seq  :", X_val_seq.shape)
print("X_test_seq :", X_test_seq.shape)

print("y_train_seq:", y_train_seq.shape)
print("y_val_seq  :", y_val_seq.shape)
print("y_test_seq :", y_test_seq.shape)

assert torch.isfinite(X_train_seq).all()
assert torch.isfinite(X_val_seq).all()
assert torch.isfinite(X_test_seq).all()

X_train_seq: torch.Size([7968, 4, 21])
X_val_seq  : torch.Size([2276, 4, 21])
X_test_seq : torch.Size([1139, 4, 21])
y_train_seq: torch.Size([7968])
y_val_seq  : torch.Size([2276])
y_test_seq : torch.Size([1139])


dataset and dataloader 

In [15]:
class SequenceBeamDataset(Dataset):
    def __init__(self, X_seq, y_seq):
        self.X_seq = X_seq
        self.y_seq = y_seq

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.y_seq[idx]

In [16]:
BATCH_SIZE = 128

train_dataset = SequenceBeamDataset(X_train_seq, y_train_seq)
val_dataset   = SequenceBeamDataset(X_val_seq, y_val_seq)
test_dataset  = SequenceBeamDataset(X_test_seq, y_test_seq)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

x_batch, y_batch = next(iter(train_loader))

print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)

seq_len = x_batch.shape[1]
input_dim = x_batch.shape[2]

print("seq_len:", seq_len)
print("input_dim:", input_dim)
print("num_classes:", num_classes)

x_batch: torch.Size([128, 4, 21])
y_batch: torch.Size([128])
seq_len: 4
input_dim: 21
num_classes: 29


evaluation functions 

top k evaluations 

In [17]:
def evaluate_topk_logits(logits, labels, ks=(1, 2, 3, 5)):
    max_k = max(ks)

    _, pred = torch.topk(logits, k=max_k, dim=1)
    pred = pred.t()

    results = {}
    total = labels.size(0)

    for k in ks:
        correct = pred[:k].eq(labels.view(1, -1)).sum().item()
        results[f"top{k}"] = 100.0 * correct / total

    return results


def evaluate_model(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    total = 0
    total_loss = 0.0
    correct = {k: 0 for k in ks}

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, labels)

            bs = labels.size(0)
            total += bs
            total_loss += loss.item() * bs

            max_k = max(ks)
            _, pred = torch.topk(logits, k=max_k, dim=1)
            pred = pred.t()

            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    metrics = {"loss": total_loss / total}

    for k in ks:
        metrics[f"top{k}"] = 100.0 * correct[k] / total

    return metrics

mlp baseline 

In [18]:
class SequenceMLP(nn.Module):
    def __init__(
        self,
        seq_len,
        input_dim,
        num_classes,
        hidden_dims=(256, 128, 64),
        dropout=0.35
    ):
        super().__init__()

        flat_dim = seq_len * input_dim

        h1, h2, h3 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(flat_dim, h1),
            nn.BatchNorm1d(h1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h3, num_classes)
        )

    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        return self.net(x)

lstm baseline 

In [19]:
class SequenceLSTM(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        embed_dim=64,
        hidden_dim=96,
        num_layers=1,
        dropout=0.35
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(96, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.classifier(last)

trainer with anti overfit control 

In [20]:
def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    device,
    epochs=100,
    lr=1e-4,
    weight_decay=1e-3,
    label_smoothing=0.05,
    patience=12,
    min_delta=1e-4,
    save_path="best_model.pth"
):
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5
    )

    best_val_loss = float("inf")
    best_val_top1 = -1
    patience_counter = 0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        train_total = 0
        train_loss_sum = 0.0
        train_correct = 0

        for x, labels in train_loader:
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            bs = labels.size(0)
            train_total += bs
            train_loss_sum += loss.item() * bs
            train_correct += (logits.argmax(dim=1) == labels).sum().item()

        train_loss = train_loss_sum / train_total
        train_acc = 100.0 * train_correct / train_total

        val_metrics = evaluate_model(model, val_loader, device, ks=(1, 2, 3, 5))
        val_loss = val_metrics["loss"]
        val_top1 = val_metrics["top1"]

        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_top1": train_acc,
            "val_loss": val_loss,
            **{f"val_{k}": v for k, v in val_metrics.items() if k != "loss"}
        }

        history.append(row)

        print(
            f"Epoch {epoch:03d} | "
            f"LR {current_lr:.2e} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Top1 {train_acc:.2f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Top1 {val_metrics['top1']:.2f} | "
            f"Top2 {val_metrics['top2']:.2f} | "
            f"Top3 {val_metrics['top3']:.2f} | "
            f"Top5 {val_metrics['top5']:.2f}"
        )

        improved = val_loss < (best_val_loss - min_delta)

        if improved:
            best_val_loss = val_loss
            best_val_top1 = val_top1
            patience_counter = 0
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    print("Best Val Loss:", best_val_loss)
    print("Best Val Top1:", best_val_top1)

    return pd.DataFrame(history)

train mlp 

In [21]:
set_seed(SEED)

mlp_model = SequenceMLP(
    seq_len=seq_len,
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(256, 128, 64),
    dropout=0.35
).to(device)

mlp_save_path = "/content/drive/MyDrive/best_sequence_mlp_clean.pth"

history_mlp = train_with_early_stopping(
    model=mlp_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100,
    lr=1e-4,
    weight_decay=1e-3,
    label_smoothing=0.05,
    patience=12,
    save_path=mlp_save_path
)

Epoch 001 | LR 1.00e-04 | Train Loss 3.2608 | Train Top1 10.68 | Val Loss 3.0139 | Val Top1 36.16 | Top2 47.63 | Top3 60.02 | Top5 68.32
Saved best model
Epoch 002 | LR 1.00e-04 | Train Loss 3.0133 | Train Top1 27.40 | Val Loss 2.8140 | Val Top1 40.03 | Top2 54.35 | Top3 65.91 | Top5 78.87
Saved best model
Epoch 003 | LR 1.00e-04 | Train Loss 2.8293 | Train Top1 33.73 | Val Loss 2.6319 | Val Top1 41.83 | Top2 58.04 | Top3 71.88 | Top5 81.63
Saved best model
Epoch 004 | LR 1.00e-04 | Train Loss 2.6686 | Train Top1 36.60 | Val Loss 2.4389 | Val Top1 41.74 | Top2 61.99 | Top3 74.17 | Top5 83.04
Saved best model
Epoch 005 | LR 1.00e-04 | Train Loss 2.5225 | Train Top1 39.90 | Val Loss 2.3289 | Val Top1 45.25 | Top2 63.22 | Top3 76.58 | Top5 85.37
Saved best model
Epoch 006 | LR 1.00e-04 | Train Loss 2.3962 | Train Top1 40.91 | Val Loss 2.2056 | Val Top1 45.91 | Top2 64.19 | Top3 79.66 | Top5 87.30
Saved best model
Epoch 007 | LR 1.00e-04 | Train Loss 2.2714 | Train Top1 42.88 | Val Loss 2.

test mlp 

In [22]:
mlp_eval = SequenceMLP(
    seq_len=seq_len,
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(256, 128, 64),
    dropout=0.35
).to(device)

mlp_eval.load_state_dict(torch.load(mlp_save_path, map_location=device))

mlp_test_metrics = evaluate_model(
    mlp_eval,
    test_loader,
    device,
    ks=(1, 2, 3, 5)
)

print("Clean Sequence MLP Test Metrics:")
print(mlp_test_metrics)

Clean Sequence MLP Test Metrics:
{'loss': 1.1170553420069345, 'top1': 61.1062335381914, 'top2': 82.70412642669008, 'top3': 90.78138718173837, 'top5': 96.57594381035996}


lstm model 

train lstm 

In [23]:
set_seed(SEED)

lstm_model = SequenceLSTM(
    input_dim=input_dim,
    num_classes=num_classes,
    embed_dim=64,
    hidden_dim=96,
    num_layers=1,
    dropout=0.35
).to(device)

lstm_save_path = "/content/drive/MyDrive/best_sequence_lstm_clean.pth"

history_lstm = train_with_early_stopping(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100,
    lr=1e-4,
    weight_decay=1e-3,
    label_smoothing=0.05,
    patience=12,
    save_path=lstm_save_path
)

Epoch 001 | LR 1.00e-04 | Train Loss 3.1169 | Train Top1 16.74 | Val Loss 2.6236 | Val Top1 32.95 | Top2 52.02 | Top3 61.25 | Top5 69.60
Saved best model
Epoch 002 | LR 1.00e-04 | Train Loss 2.5508 | Train Top1 34.41 | Val Loss 2.2090 | Val Top1 39.06 | Top2 59.27 | Top3 69.64 | Top5 79.35
Saved best model
Epoch 003 | LR 1.00e-04 | Train Loss 2.2789 | Train Top1 38.97 | Val Loss 1.9785 | Val Top1 43.23 | Top2 62.65 | Top3 74.38 | Top5 83.92
Saved best model
Epoch 004 | LR 1.00e-04 | Train Loss 2.1024 | Train Top1 42.43 | Val Loss 1.8208 | Val Top1 45.52 | Top2 65.07 | Top3 77.07 | Top5 87.61
Saved best model
Epoch 005 | LR 1.00e-04 | Train Loss 1.9786 | Train Top1 44.43 | Val Loss 1.7036 | Val Top1 47.67 | Top2 68.67 | Top3 79.31 | Top5 89.72
Saved best model
Epoch 006 | LR 1.00e-04 | Train Loss 1.8903 | Train Top1 46.13 | Val Loss 1.6179 | Val Top1 48.81 | Top2 70.83 | Top3 81.28 | Top5 89.85
Saved best model
Epoch 007 | LR 1.00e-04 | Train Loss 1.8144 | Train Top1 47.87 | Val Loss 1.

test lstm 

In [24]:
lstm_eval = SequenceLSTM(
    input_dim=input_dim,
    num_classes=num_classes,
    embed_dim=64,
    hidden_dim=96,
    num_layers=1,
    dropout=0.35
).to(device)

lstm_eval.load_state_dict(torch.load(lstm_save_path, map_location=device))

lstm_test_metrics = evaluate_model(
    lstm_eval,
    test_loader,
    device,
    ks=(1, 2, 3, 5)
)

print("Clean Sequence LSTM Test Metrics:")
print(lstm_test_metrics)

Clean Sequence LSTM Test Metrics:
{'loss': 1.0239943169626466, 'top1': 61.018437225636525, 'top2': 83.66988586479368, 'top3': 92.09833187006146, 'top5': 97.1027216856892}


In [25]:
results = pd.DataFrame([
    {
        "Model": "Sequence MLP",
        "Test Loss": mlp_test_metrics["loss"],
        "Top-1": mlp_test_metrics["top1"],
        "Top-2": mlp_test_metrics["top2"],
        "Top-3": mlp_test_metrics["top3"],
        "Top-5": mlp_test_metrics["top5"],
    },
    {
        "Model": "Sequence LSTM",
        "Test Loss": lstm_test_metrics["loss"],
        "Top-1": lstm_test_metrics["top1"],
        "Top-2": lstm_test_metrics["top2"],
        "Top-3": lstm_test_metrics["top3"],
        "Top-5": lstm_test_metrics["top5"],
    }
])

display(results)

,Model,Test Loss,Top-1,Top-2,Top-3,Top-5
0,Sequence MLP,1.117055,61.106234,82.704126,90.781387,96.575944
1,Sequence LSTM,1.023994,61.018437,83.669886,92.098332,97.102722
